# Audio Spectrogram Transformer (AST)

THe reason is our CNN (EfficientNet-B0) only looks at local spectral patterns. But genre also depends on long-range temporal structure like chord progressions and rhythm patterns across the full 10-second clip. The AST uses self-attention to capture these global dependencies.

Since AST is pretrained on AudioSet (5800 hours, 527 audio classes), so it already understands audio really well. We just fine-tune it on our genre task.

## 1. Setup and Imports

In [ ]:
!pip install librosa transformers wandb --quiet

import os
import glob
import random
import warnings
import time
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import librosa
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")

## 2. Configuration

In [ ]:
# paths
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
OUTPUT_DIR = "./outputs"

STEMS_DIR = os.path.join(DATA_ROOT, "genres_stems")
NOISE_DIR = os.path.join(DATA_ROOT, "ESC-50-master", "audio")
TEST_DIR  = os.path.join(DATA_ROOT, "mashups")
TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")

# AST expects 16kHz audio (different from CNN which used 22050)
SR = 16000
DURATION = 10.0
TARGET_LEN = int(SR * DURATION)

# genres
GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                 'jazz', 'metal', 'pop', 'reggae', 'rock'])
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}
STEMS = ['drums', 'vocals', 'bass']  # 'others' missing for all songs

# training params
# AST is much bigger than EfficientNet so we use smaller batch size
SAMPLES_PER_GENRE = 800     # 8k mashups per epoch
BATCH_SIZE = 8               # AST needs lots of memory
ACCUM_STEPS = 4              # effective batch = 8 * 4 = 32
EPOCHS = 20
LR_BACKBONE = 1e-5           # low LR to preserve pretrained weights
LR_HEAD = 1e-3               # high LR for new classification head
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
NUM_WORKERS = 4
WARMUP_EPOCHS = 2

# stem weights from EDA - drums most informative
STEM_WEIGHTS = {'drums': 0.45, 'vocals': 0.35, 'bass': 0.20}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Will train on {SAMPLES_PER_GENRE * 10} mashups/epoch")
print(f"Effective batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}")

## 3. Weights & Biases

In [ ]:
import wandb

wandb.login(key="wandb-api-key")

run = wandb.init(
    entity="23f3003225-indian-institute-of-technology-madras",
    project="23f3003225-dl-genai-project",
    name="exp_004_ast",
    config={
        "backbone": "MIT/ast-finetuned-audioset-10-10-0.4593",
        "samples_per_genre": SAMPLES_PER_GENRE,
        "batch_size": BATCH_SIZE * ACCUM_STEPS,
        "epochs": EPOCHS,
        "lr_backbone": LR_BACKBONE,
        "lr_head": LR_HEAD,
    },
    tags=["ast", "transformer", "audioset"],
)

## 4. Build Data Index

Same as the CNN experiment - scan all genre folders and split 85/15.

In [ ]:
# index all stem files
stem_index = {g: {st: [] for st in STEMS} for g in GENRES}
song_index = {g: [] for g in GENRES}

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)
    songs = sorted(s for s in os.listdir(genre_path)
                   if os.path.isdir(os.path.join(genre_path, s)))
    
    for song in songs:
        song_dir = os.path.join(genre_path, song)
        available_stems = []
        for stem in STEMS:
            filepath = os.path.join(song_dir, f"{stem}.wav")
            if os.path.exists(filepath):
                stem_index[genre][stem].append(filepath)
                available_stems.append(stem)
        if available_stems:
            song_index[genre].append({
                'dir': song_dir,
                'stems': available_stems
            })

noise_files = sorted(glob.glob(os.path.join(NOISE_DIR, "*.wav")))
print(f"Found {len(noise_files)} ESC-50 noise clips")

In [ ]:
# train/val split (85/15)
train_stems = {g: {st: [] for st in STEMS} for g in GENRES}
val_songs = {g: [] for g in GENRES}

for genre in GENRES:
    songs = song_index[genre].copy()
    random.shuffle(songs)
    split_point = int(0.85 * len(songs))
    
    train_list = songs[:split_point]
    val_list = songs[split_point:]
    val_songs[genre] = val_list
    
    train_dirs = {s['dir'] for s in train_list}
    for stem in STEMS:
        train_stems[genre][stem] = [
            fp for fp in stem_index[genre][stem]
            if os.path.dirname(fp) in train_dirs
        ]
    
    print(f"  {genre}: {len(train_list)} train, {len(val_list)} val")

## 5. Audio Loading

Note: AST uses 16kHz (not 22050 like our CNN). The AST feature extractor handles mel conversion internally, so we just need to load raw waveforms.

In [ ]:
def load_wav(path, sr=SR, target_len=TARGET_LEN):
    # load audio and pad/crop to fixed length
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        wav = torch.from_numpy(y).float()
        if len(wav) < target_len:
            wav = F.pad(wav, (0, target_len - len(wav)))
        elif len(wav) > target_len:
            start = random.randint(0, len(wav) - target_len)
            wav = wav[start:start + target_len]
        return wav
    except:
        return torch.zeros(target_len)

print("Audio loading ready")

## 6. Datasets

Same mashup augmentation as CNN - mix stems from different songs, add noise, apply overdrive. The only difference is we return raw waveforms instead of mel spectrograms (AST handles that).

In [ ]:
class MashupDataset(Dataset):
    # generates synthetic mashups on-the-fly
    
    def __init__(self, stem_idx, noise_files, samples_per_genre=800, augment=True):
        self.stem_idx = stem_idx
        self.noise_files = noise_files
        self.augment = augment
        self.samples = []
        for genre in GENRES:
            for _ in range(samples_per_genre):
                self.samples.append(GENRE2IDX[genre])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre_idx = self.samples[idx]
        genre = IDX2GENRE[genre_idx]

        # pick random stems from different songs of same genre
        stems_wav = []
        for stem_type in STEMS:
            available = self.stem_idx[genre][stem_type]
            if not available:
                continue
            wav = load_wav(random.choice(available))
            # weight by stem importance
            gain = random.uniform(0.5, 1.5) * (STEM_WEIGHTS[stem_type] / 0.33)
            stems_wav.append(wav * gain)

        if not stems_wav:
            return torch.zeros(TARGET_LEN), genre_idx

        # mix all stems
        mix = torch.stack(stems_wav).sum(0)

        if self.augment:
            # circular time shift
            mix = torch.roll(mix, random.randint(-SR, SR))
            
            # add ESC-50 noise
            for _ in range(random.randint(0, 2)):
                noise = load_wav(random.choice(self.noise_files))
                snr_db = random.uniform(5.0, 25.0)
                sig_pwr = mix.pow(2).mean() + 1e-10
                nse_pwr = noise.pow(2).mean() + 1e-10
                scale = (sig_pwr / (nse_pwr * 10 ** (snr_db / 10))).sqrt()
                mix = mix + noise * scale
            
            # overdrive (30% chance)
            if random.random() < 0.3:
                mix = torch.clamp(mix * random.uniform(1.2, 3.0), -1, 1)

        # normalize
        peak = mix.abs().max()
        if peak > 1e-6:
            mix = mix / peak * random.uniform(0.7, 1.0)
        
        return mix, genre_idx

In [ ]:
class ValDataset(Dataset):
    # validation - just mix stems from each song, no augmentation
    
    def __init__(self, song_index):
        self.items = []
        for genre in GENRES:
            for song in song_index[genre]:
                self.items.append((song, GENRE2IDX[genre]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        song_info, label = self.items[idx]
        stems = [load_wav(os.path.join(song_info['dir'], f"{st}.wav"))
                 for st in song_info['stems']]
        mix = torch.stack(stems).sum(0)
        peak = mix.abs().max()
        if peak > 1e-6:
            mix = mix / peak
        return mix, label


class TestDataset(Dataset):
    # test mashups
    
    def __init__(self, test_dir, test_csv):
        self.df = pd.read_csv(test_csv, dtype={'id': str})
        self.paths = []
        for _, row in self.df.iterrows():
            path = None
            for pattern in [f"song{str(row['id']).zfill(4)}.wav",
                          f"{row['id']}.wav",
                          f"song{row['id']}.wav"]:
                p = os.path.join(test_dir, pattern)
                if os.path.exists(p):
                    path = p
                    break
            self.paths.append(path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.paths[idx]
        if path:
            wav = load_wav(path)
        else:
            wav = torch.zeros(TARGET_LEN)
        return wav, str(self.df.iloc[idx]['id'])

print("Datasets ready")

## 7. AST Model

The Audio Spectrogram Transformer from MIT, pretrained on AudioSet (527 audio classes). We replace the classification head with a 10-class head for our genres.

The feature extractor converts raw waveforms into the format AST expects. We use collator functions to do this in the dataloader.

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

# this handles converting raw audio to mel spectrograms for AST
feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)
print("Feature extractor loaded")

In [ ]:
class ASTGenreClassifier(nn.Module):
    # wrapper around HuggingFace AST model
    
    def __init__(self, num_classes=10):
        super().__init__()
        self.ast = ASTForAudioClassification.from_pretrained(
            AST_MODEL_NAME,
            num_labels=num_classes,
            ignore_mismatched_sizes=True  # head size changes from 527 -> 10
        )

    def forward(self, input_values):
        # input_values shape: (batch, 1024, 128) - from feature extractor
        outputs = self.ast(input_values=input_values)
        return outputs.logits

model = ASTGenreClassifier(num_classes=10).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"AST parameters: {total_params / 1e6:.1f}M")

## 8. Collator Functions

The AST feature extractor needs to run on batches of raw waveforms. We use custom collator functions to do this inside the DataLoader.

In [ ]:
class ASTCollator:
    # converts raw waveforms to AST input format during batching
    
    def __init__(self, feature_extractor, sr=16000):
        self.fe = feature_extractor
        self.sr = sr

    def __call__(self, batch):
        waveforms, labels = zip(*batch)
        
        # run feature extractor on numpy arrays
        waveforms_np = [w.numpy() for w in waveforms]
        inputs = self.fe(
            waveforms_np,
            sampling_rate=self.sr,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )
        
        # handle both int labels (train/val) and string IDs (test)
        if isinstance(labels[0], int) or isinstance(labels[0], np.integer):
            labels_out = torch.tensor(labels, dtype=torch.long)
        else:
            labels_out = labels
        
        return inputs["input_values"], labels_out


class ASTTestCollator:
    # same but for test set where labels are string IDs
    
    def __init__(self, feature_extractor, sr=16000):
        self.fe = feature_extractor
        self.sr = sr

    def __call__(self, batch):
        waveforms, ids = zip(*batch)
        waveforms_np = [w.numpy() for w in waveforms]
        inputs = self.fe(
            waveforms_np,
            sampling_rate=self.sr,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )
        return inputs["input_values"], list(ids)

print("Collators ready")

## 9. Training and Evaluation Functions

Key difference from CNN: we use gradient accumulation since AST only fits batch_size=8 in GPU memory. We accumulate gradients over 4 steps to get an effective batch of 32.

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0
    num_samples = 0
    optimizer.zero_grad()

    for step, (input_values, labels) in enumerate(tqdm(loader, desc="Train", leave=False)):
        input_values = input_values.to(DEVICE)
        labels = labels.to(DEVICE)

        with autocast():
            logits = model(input_values)
            # divide loss by accum steps since we accumulate gradients
            loss = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        # only update weights every ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS * len(labels)
        num_samples += len(labels)

    return total_loss / num_samples


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    for input_values, labels in loader:
        input_values = input_values.to(DEVICE)
        with autocast():
            logits = model(input_values)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return f1, acc, np.array(all_preds), np.array(all_labels)

## 10. Optimizer with Differential Learning Rate

This is important - the AST backbone is already pretrained on AudioSet, so we use a very low LR (1e-5) to not destroy those learned representations. The new classification head gets a higher LR (1e-3) since it needs to learn from scratch.

We also use a cosine schedule with 2 epochs of warmup to let the head stabilize before the backbone starts changing.

In [ ]:
# separate backbone and head parameters
backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

print(f"Backbone: {sum(p.numel() for p in backbone_params) / 1e6:.1f}M params (lr={LR_BACKBONE})")
print(f"Head: {sum(p.numel() for p in head_params)} params (lr={LR_HEAD})")

# different learning rates for backbone vs head
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# cosine schedule with warmup
def get_lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        # linear warmup
        return (epoch + 1) / WARMUP_EPOCHS
    # cosine decay after warmup
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = GradScaler()

## 11. Training Loop

In [ ]:
collator = ASTCollator(feature_extractor, sr=SR)

val_ds = ValDataset(val_songs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        collate_fn=collator)
print(f"Validation samples: {len(val_ds)}")

best_f1 = 0.0
history = {'loss': [], 'val_f1': [], 'val_acc': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # new mashups every epoch
    train_ds = MashupDataset(train_stems, noise_files, SAMPLES_PER_GENRE, augment=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              drop_last=True, collate_fn=collator)

    loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
    scheduler.step()
    
    val_f1, val_acc, _, _ = evaluate(model, val_loader)
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - start_time

    # track history
    history['loss'].append(loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)
    wandb.log({"epoch": epoch, "loss": loss, "val_f1": val_f1,
               "val_acc": val_acc, "lr": lr})

    # save best
    tag = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_ast.pth'))
        tag = " [best]"

    print(f"E{epoch:02d}/{EPOCHS} | loss={loss:.4f} | f1={val_f1:.4f} | "
          f"acc={val_acc:.4f} | lr={lr:.6f} | {elapsed:.0f}s{tag}")

print(f"\nBest validation F1: {best_f1:.4f}")

## 12. Results

In [ ]:
# training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_f1'], label='F1')
axes[1].plot(history['val_acc'], label='Accuracy', alpha=0.7)
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

plt.suptitle(f'AST - Best F1: {best_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ast_curves.png'), dpi=150)
wandb.log({"plots/ast_curves": wandb.Image(fig)})
plt.show()

In [ ]:
# confusion matrix
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_ast.pth'), weights_only=True))
val_f1, val_acc, preds, labels = evaluate(model, val_loader)

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=GENRES).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'AST Confusion Matrix - F1={val_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ast_confusion.png'), dpi=150)
wandb.log({"plots/ast_confusion": wandb.Image(fig)})
plt.show()

print(classification_report(labels, preds, target_names=GENRES))

## 13. Inference with TTA

Run inference 5 times and average predictions. Since AST processes the full spectrogram deterministically, TTA here mainly helps with numerical stability.

In [ ]:
@torch.no_grad()
def predict_tta(model, loader, n_tta=5):
    # run inference multiple times and average
    model.eval()
    
    # collect all IDs first
    all_ids = []
    for _, ids in loader:
        all_ids.extend(ids)

    # run n_tta forward passes
    all_probs = []
    for tta_round in range(n_tta):
        round_probs = []
        for input_values, _ in loader:
            input_values = input_values.to(DEVICE)
            with autocast():
                logits = model(input_values)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            round_probs.append(probs)
        all_probs.append(np.vstack(round_probs))

    # average across rounds
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs.argmax(1), all_ids, avg_probs


# run on test set
test_collator = ASTTestCollator(feature_extractor, sr=SR)
test_ds = TestDataset(TEST_DIR, TEST_CSV)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=test_collator)
print(f"Test samples: {len(test_ds)}")

preds, ids, probs = predict_tta(model, test_loader, n_tta=5)
print(f"Prediction distribution: {Counter(preds)}")

## 14. Generate Submission

In [ ]:
# create submission csv
test_df = pd.read_csv(TEST_CSV, dtype={'id': str})
pred_dict = {str(id_): IDX2GENRE[p] for id_, p in zip(ids, preds)}
test_df['genre'] = test_df['id'].apply(lambda x: pred_dict.get(str(x), 'rock'))
test_df[['id', 'genre']].to_csv(os.path.join(OUTPUT_DIR, 'submission_ast.csv'), index=False)

print("Submission saved!")
print(test_df['genre'].value_counts().sort_index())

# save probabilities for later ensemble with CNN
np.save(os.path.join(OUTPUT_DIR, 'test_probs_ast.npy'), probs)
print("\nProbabilities saved for ensemble use")

## 15. Finish

In [ ]:
wandb.log({"best_f1": best_f1, "status": "complete"})

# save model artifact
artifact = wandb.Artifact("ast_model", type="model")
artifact.add_file(os.path.join(OUTPUT_DIR, 'best_ast.pth'))
run.log_artifact(artifact)

wandb.finish()
print(f"Training complete. best val_f1= {best_f1:.4f}")